Input and Output Parameter Configuration

In [0]:
dbutils.widgets.text("input_path", "")
dbutils.widgets.text("output_path", "")

ADLS Congiguration for data access

In [0]:
storage_account = "adlsstoragedevesh"
account_key = "********************************************"
storage_account_url = "abfss://bronzelayer@adlsstoragedevesh.dfs.core.windows.net/bronze/"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    account_key
)

Column name formatting and saving to silverlayer container for Customer table

In [0]:
from pyspark.sql.functions import col
import re

bronze_path = "abfss://bronzelayer@adlsstoragedevesh.dfs.core.windows.net/bronze/Customer_data.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)


def clean_column_names(df):
    for col_name in df.columns:
        new_col = re.sub(r'[ ,;{}()\n\t=]', '_', col_name) 
        df = df.withColumnRenamed(col_name, new_col)
    return df

df_clean = clean_column_names(df)


silver_path = "abfss://silverlayer@adlsstoragedevesh.dfs.core.windows.net/customer"

df_clean.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)


print("Row Count:", df_clean.count())
display(df_clean.limit(10))

Row Count: 18485


CustomerKey,Customer_ID,Customer,City,State-Province,Country-Region,Postal_Code
-1,[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable],[Not Applicable]
11000,AW00011000,Jon Yang,Rockhampton,Queensland,Australia,4700
11001,AW00011001,Eugene Huang,Seaford,Victoria,Australia,3198
11002,AW00011002,Ruben Torres,Hobart,Tasmania,Australia,7001
11003,AW00011003,Christy Zhu,North Ryde,New South Wales,Australia,2113
11004,AW00011004,Elizabeth Johnson,Wollongong,New South Wales,Australia,2500
11005,AW00011005,Julio Ruiz,East Brisbane,Queensland,Australia,4169
11006,AW00011006,Janet Alvarez,Matraville,New South Wales,Australia,2036
11007,AW00011007,Marco Mehta,Warrnambool,Victoria,Australia,3280
11008,AW00011008,Rob Verhoff,Bendigo,Victoria,Australia,3550


Data cleaning and saving for the rest of the tables

In [0]:
from pyspark.sql.functions import col, regexp_replace
import re


def clean_column_names(df):
    for col_name in df.columns:
        new_col = re.sub(r'[ ,;{}()\n\t=]', '_', col_name)
        df = df.withColumnRenamed(col_name, new_col)
    return df


def clean_currency_columns(df):
    for c in df.columns:
        df = df.withColumn(c, regexp_replace(col(c), "[$,]", ""))
    return df


bronze_base = "abfss://bronzelayer@adlsstoragedevesh.dfs.core.windows.net/bronze/"


files = {
    "Product_data.csv": "product",
    "Reseller_data.csv": "reseller",
    "Sales_data.csv": "sales",
    "Sales Order_data.csv": "sales_order",
    "Sales Territory_data.csv": "sales_territory",
    "Date_data.csv": "date"
}


silver_base = "abfss://silverlayer@adlsstoragedevesh.dfs.core.windows.net/"


for file_name, table_name in files.items():
    print(f"\nProcessing: {file_name}")

    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(bronze_base + file_name)

    df = clean_column_names(df)

    df = clean_currency_columns(df)

    df.write.format("delta") \
        .mode("overwrite") \
        .save(silver_base + table_name)

    print(f"{table_name} loaded successfully with {df.count()} rows")


Processing: Product_data.csv
product loaded successfully with 397 rows

Processing: Reseller_data.csv
reseller loaded successfully with 702 rows

Processing: Sales_data.csv
sales loaded successfully with 121253 rows

Processing: Sales Order_data.csv
sales_order loaded successfully with 121253 rows

Processing: Sales Territory_data.csv
sales_territory loaded successfully with 11 rows

Processing: Date_data.csv
date loaded successfully with 1461 rows
